# 01 Validate and Merge Dataset

Validate teammate-provided YOLO folders, merge valid samples into the merged dataset at `data/master_original`, and optionally ingest a separate held-out test set into `data/splits_original/test`.


## Purpose

This notebook validates source folders, saves reports, warns about invalid or suspicious samples, and copies valid image-label pairs with normalized filenames. It does not split training data, augment data, train models, or modify raw source folders.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import yaml
from IPython.display import display


def find_v2_root(start: Path) -> Path:
    """Find the v2_pipeline root from the current notebook directory."""
    for candidate in (start, *start.parents):
        if candidate.name == "v2_pipeline" and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate the v2_pipeline root.")


V2_ROOT = find_v2_root(Path.cwd().resolve())
PROJECT_ROOT = V2_ROOT.parent.parent
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

from src.dataset.merge_dataset import merge_valid_samples
from src.dataset.validate_yolo_dataset import validate_dataset

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"V2_ROOT={V2_ROOT}")

PROJECT_ROOT=C:\Github\smart-factory-safety-monitoring-system
V2_ROOT=C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline


## Load Configuration


In [2]:
config_path = V2_ROOT / "configs" / "dataset_config.yaml"
with config_path.open("r", encoding="utf-8") as file_handle:
    dataset_config = yaml.safe_load(file_handle)

raw_sources_dir = V2_ROOT / dataset_config["raw_sources_dir"]
test_sources_dir = V2_ROOT / dataset_config["test_sources_dir"]
merged_dataset_dir = V2_ROOT / dataset_config["master_original_dir"]
splits_original_dir = V2_ROOT / dataset_config["splits_original_dir"]

merged_images_dir = merged_dataset_dir / "images"
merged_labels_dir = merged_dataset_dir / "labels"
test_images_dir = splits_original_dir / "test" / "images"
test_labels_dir = splits_original_dir / "test" / "labels"
validation_reports_dir = V2_ROOT / "reports" / "validation"
validation_reports_dir.mkdir(parents=True, exist_ok=True)

class_ids = {int(class_id) for class_id in dataset_config["classes"].keys()}

print(f"Raw sources: {raw_sources_dir}")
print(f"Optional test sources: {test_sources_dir}")
print(f"Merged dataset images: {merged_images_dir}")
print(f"Merged dataset labels: {merged_labels_dir}")
print(f"Test images: {test_images_dir}")
print(f"Test labels: {test_labels_dir}")
print(f"Class IDs: {sorted(class_ids)}")

Raw sources: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\raw_sources
Optional test sources: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\test_sources
Merged dataset images: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\master_original\images
Merged dataset labels: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\master_original\labels
Test images: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\splits_original\test\images
Test labels: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\splits_original\test\labels
Class IDs: [0, 1, 2]


## Validate Raw Sources


In [3]:
validation_df = validate_dataset(raw_sources_dir, class_ids=class_ids)

validation_report_path = validation_reports_dir / "validation_report.csv"
invalid_samples_path = validation_reports_dir / "invalid_samples.csv"

validation_df.to_csv(validation_report_path, index=False)
invalid_df = validation_df.loc[validation_df["status"].ne("valid")].copy()
invalid_df.to_csv(invalid_samples_path, index=False)

print(f"Saved validation report: {validation_report_path}")
print(f"Saved invalid samples report: {invalid_samples_path}")
display(validation_df.head(20))

Saved validation report: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\validation\validation_report.csv
Saved invalid samples report: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\validation\invalid_samples.csv


,source,base_name,image_path,label_path,status,errors,warnings,image_width,image_height,num_objects,num_person,num_helmet,num_vest
0,Hoang,ppe_0001,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,2676,1568,6,2,2,2
1,Hoang,ppe_0002,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,2676,1568,21,7,7,7
2,Hoang,ppe_0003,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,1361,784,12,4,4,4
3,Hoang,ppe_0004,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,1338,784,3,1,1,1
4,Hoang,ppe_0005,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,2594,1632,5,2,2,1
5,Hoang,ppe_0006,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,2594,1632,4,1,2,1
6,Hoang,ppe_0007,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,2594,1632,5,1,2,2
7,Hoang,ppe_0008,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,2594,1632,6,2,2,2
8,Hoang,ppe_0009,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,2594,1632,6,2,2,2
9,Hoang,ppe_0010,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,,2594,1632,6,2,2,2


## Dataset Summary


In [4]:
source_folders_found = 0
if raw_sources_dir.exists():
    source_folders_found = sum(1 for path in raw_sources_dir.iterdir() if path.is_dir())

warning_rows = (
    validation_df["warnings"].fillna("").ne("") if not validation_df.empty else []
)
cross_source_duplicate_rows = 0
if not validation_df.empty:
    cross_source_duplicate_rows = (
        validation_df["warnings"]
        .fillna("")
        .str.contains(
            "Duplicate base name appears across sources",
            regex=False,
        )
        .sum()
    )

dataset_summary = pd.DataFrame(
    [
        {
            "source_folders_found": source_folders_found,
            "total_rows": len(validation_df),
            "valid_rows": int(validation_df["status"].eq("valid").sum())
            if not validation_df.empty
            else 0,
            "invalid_rows": int(validation_df["status"].ne("valid").sum())
            if not validation_df.empty
            else 0,
            "warning_rows": int(warning_rows.sum()) if not validation_df.empty else 0,
            "cross_source_duplicate_rows": int(cross_source_duplicate_rows),
            "total_objects": int(validation_df["num_objects"].sum())
            if not validation_df.empty
            else 0,
            "total_person": int(validation_df["num_person"].sum())
            if not validation_df.empty
            else 0,
            "total_helmet": int(validation_df["num_helmet"].sum())
            if not validation_df.empty
            else 0,
            "total_vest": int(validation_df["num_vest"].sum())
            if not validation_df.empty
            else 0,
        }
    ]
)

dataset_summary_path = validation_reports_dir / "dataset_summary.csv"
dataset_summary.to_csv(dataset_summary_path, index=False)

print(f"Saved dataset summary: {dataset_summary_path}")
if not validation_df.empty:
    display(
        validation_df.groupby(["source", "status"]).size().reset_index(name="count")
    )
display(dataset_summary)

Saved dataset summary: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\validation\dataset_summary.csv


,source,status,count
0,Hoang,valid,124
1,Tri,valid,195
2,Yen,valid,156


,source_folders_found,total_rows,valid_rows,invalid_rows,warning_rows,cross_source_duplicate_rows,total_objects,total_person,total_helmet,total_vest
0,3,475,475,0,0,0,3203,1308,834,1061


## Review Validation Findings


In [5]:
has_rows = not validation_df.empty
valid_rows_count = int(validation_df["status"].eq("valid").sum()) if has_rows else 0
invalid_rows_count = int(validation_df["status"].ne("valid").sum()) if has_rows else 0

if not raw_sources_dir.exists():
    print(f"No raw source directory found yet: {raw_sources_dir}")
elif not has_rows:
    print(
        "No source samples found. Add teammate folders under data/raw_sources before merging."
    )
else:
    print(f"Valid rows: {valid_rows_count}")
    print(f"Invalid rows: {invalid_rows_count}")

    if invalid_rows_count:
        print("Invalid samples were found. They will be reported but not merged.")
        display(invalid_df[["source", "base_name", "errors"]].head(20))

    warning_df = validation_df.loc[validation_df["warnings"].fillna("").ne("")]
    if not warning_df.empty:
        print(
            "Warnings were found. Review them before moving to split or training steps."
        )
        display(warning_df[["source", "base_name", "warnings"]].head(20))

Valid rows: 475
Invalid rows: 0


## Merge Valid Original Samples


In [6]:
if valid_rows_count == 0:
    merge_report_df = pd.DataFrame()
    print(
        "No valid samples to merge. Resolve validation issues and rerun this notebook."
    )
else:
    merge_report_df = merge_valid_samples(
        validation_df=validation_df,
        output_images_dir=merged_images_dir,
        output_labels_dir=merged_labels_dir,
        prefix="ppe",
    )

merge_report_path = validation_reports_dir / "merge_report.csv"
merge_report_df.to_csv(merge_report_path, index=False)

print(f"Saved merge report: {merge_report_path}")
if not merge_report_df.empty:
    print(merge_report_df["merge_status"].value_counts().to_string())
display(merge_report_df.head(20))

Saved merge report: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\validation\merge_report.csv
merge_status
merged    475


,source,base_name,original_image_path,original_label_path,new_image_path,new_label_path,merge_status,notes
0,Hoang,ppe_0001,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,
1,Hoang,ppe_0002,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,
2,Hoang,ppe_0003,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,
3,Hoang,ppe_0004,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,
4,Hoang,ppe_0005,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,
5,Hoang,ppe_0006,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,
6,Hoang,ppe_0007,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,
7,Hoang,ppe_0008,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,
8,Hoang,ppe_0009,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,
9,Hoang,ppe_0010,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,


## Optional Separate Test Set


In [7]:
test_source_folders = []
if test_sources_dir.exists():
    test_source_folders = [path for path in test_sources_dir.iterdir() if path.is_dir()]

if not test_source_folders:
    test_validation_df = pd.DataFrame()
    test_invalid_df = pd.DataFrame()
    test_merge_report_df = pd.DataFrame()
    print(
        "No separate test sources found. Add folders under data/test_sources if needed."
    )
else:
    test_validation_df = validate_dataset(test_sources_dir, class_ids=class_ids)
    test_invalid_df = test_validation_df.loc[
        test_validation_df["status"].ne("valid")
    ].copy()
    test_valid_rows_count = int(test_validation_df["status"].eq("valid").sum())

    test_validation_report_path = validation_reports_dir / "test_validation_report.csv"
    test_invalid_samples_path = validation_reports_dir / "test_invalid_samples.csv"
    test_validation_df.to_csv(test_validation_report_path, index=False)
    test_invalid_df.to_csv(test_invalid_samples_path, index=False)

    print(f"Saved test validation report: {test_validation_report_path}")
    print(f"Saved test invalid samples report: {test_invalid_samples_path}")
    print(f"Valid test rows: {test_valid_rows_count}")
    print(f"Invalid test rows: {len(test_invalid_df)}")

    if not test_invalid_df.empty:
        display(test_invalid_df[["source", "base_name", "errors"]].head(20))

    if test_valid_rows_count == 0:
        test_merge_report_df = pd.DataFrame()
        print("No valid separate test samples to merge.")
    else:
        test_merge_report_df = merge_valid_samples(
            validation_df=test_validation_df,
            output_images_dir=test_images_dir,
            output_labels_dir=test_labels_dir,
            prefix="test",
        )

test_merge_report_path = validation_reports_dir / "test_merge_report.csv"
test_merge_report_df.to_csv(test_merge_report_path, index=False)
print(f"Saved test merge report: {test_merge_report_path}")
display(test_merge_report_df.head(20))

Saved test validation report: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\validation\test_validation_report.csv
Saved test invalid samples report: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\validation\test_invalid_samples.csv
Valid test rows: 50
Invalid test rows: 0
Saved test merge report: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\validation\test_merge_report.csv


,source,base_name,original_image_path,original_label_path,new_image_path,new_label_path,merge_status,notes
0,Tuan,ppe_1,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,
1,Tuan,ppe_10,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,
2,Tuan,ppe_11,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,
3,Tuan,ppe_12,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,
4,Tuan,ppe_13,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,
5,Tuan,ppe_14,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,
6,Tuan,ppe_15,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,
7,Tuan,ppe_16,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,
8,Tuan,ppe_17,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,
9,Tuan,ppe_18,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,merged,


## Final Checklist

- `validation_report.csv` saved under `reports/validation/`.
- `invalid_samples.csv` saved for rows that will not be merged.
- `dataset_summary.csv` saved with validation totals and object counts.
- `merge_report.csv` saved after copying valid originals as `ppe_00001`, `ppe_00002`, ...
- Optional separate test reports saved if `data/test_sources` contains source folders.
- Optional valid test samples copied as `test_00001`, `test_00002`, ...
- No train/val split, augmentation, training, or legacy edits were performed.
